# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # mlcroissant's object, not a dict

print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print("\nDataset Keywords:")
pprint.pprint(getattr(metadata, 'keywords', []))

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we list the available Record Sets, their `@id`s, and associated fields (by their `@id`).
The `@id` is used for referencing in all future steps.

In [ ]:
record_sets = getattr(metadata, 'recordSet', [])

if not record_sets:
    print("No record sets found in the Croissant metadata.")
else:
    for record_set in record_sets:
        print(f"RecordSet @id: {record_set['@id']}")
        rs = dataset._find_by_id(record_set['@id'])  # Internal: Extract the RecordSet object
        fields = getattr(rs, 'field', [])
        if not fields:
            print("  (No fields listed)")
        else:
            for field in fields:
                print(f"   Field @id: {field['@id']}")
        print()
if not record_sets:
    print("You may need to manually inspect the dataset's data files for structure, as no RecordSet is declared.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

**Note**: For this dataset, the Croissant schema does not directly declare any `recordSet`. 
However, `mlcroissant` will attempt to extract DataFrames from the main data file(s) listed in the distribution section.

In [ ]:
# Attempt to identify all available (even implicit) record sets via dataset.get_record_set_ids()
try:
    available_record_sets = dataset.get_record_set_ids()
except Exception:
    available_record_sets = []

if available_record_sets:
    print("Record sets detected:")
    for recset in available_record_sets:
        print(f"  {recset}")
else:
    print("No explicit record sets found; defaulting to extracting all tabular files listed in 'distribution'.")

# Extract the main tabular DataFrames (uses the distribution section)
dataframes = {}
for rec in dataset.records():
    """
    The dataset.records() generator will yield dicts for each record from the available files.
    We collect them into a list of dicts (DataFrame),
    using the first Distribution file as the implicit record set.
    """
    dataframes.setdefault('main_data', []).append(rec)
    if len(dataframes['main_data']) >= 5:
        # Limit preview for demonstration: only collect first 5 for print
        break

if 'main_data' in dataframes:
    df = pd.DataFrame(dataframes['main_data'])
    print(f"Extracted columns: {df.columns.tolist()}")
    display(df.head())
else:
    print("No rows/records found in main data distribution.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Since the Croissant schema does not declare record set and field structures, we demonstrate these steps using likely protein-related fields commonly present, e.g., `MW` (Molecular Weight), `Coverage_pct`, and `Description` or `ProteinGroup`.

Update field names and `@id` references as needed for your downstream work.

In [ ]:
# Suppose our DataFrame 'df' has columns named 'MW', 'Coverage_pct', 'Description'
if 'MW' in df.columns:
    numeric_field = 'MW'  # Example numeric field (@id: MW)
    threshold = 50000  # For illustration, e.g., MW > 50kDa
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize the MW field
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Grouping (if there is a descriptive or categorical field)
    group_field = "Description" if "Description" in df.columns else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        display(grouped_df.head())
    else:
        print("No group field like 'Description' found for grouping.")
else:
    print("No column 'MW' found; please inspect df.columns and update the notebook with a valid numeric field.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

The example below plots the distribution of protein molecular weights (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'MW' in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df['MW'].dropna(), bins=30, kde=True)
    plt.title('Distribution of Protein Molecular Weight (MW)')
    plt.xlabel('Molecular Weight (Da)')
    plt.ylabel('Frequency')
    plt.show()
else:
    print("'MW' column not found for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to load and explore the FAIR² dataset using the `mlcroissant` library. 
We previewed the metadata, attempted to extract main record data, performed basic EDA (filtering and normalization), and visualized the distribution of protein molecular weights.

For advanced analyses, refer to field and column `@id` references from the Croissant metadata as available.

**Note**: If the dataset schema is updated to explicitly declare more `recordSet` or field `@id`s, update the notebook accordingly to reference those entities by ID.